# **Basic Tasks**

In [0]:
%sql
CREATE OR REPLACE TABLE dev.demo.demo7
(
    id INT,
    name STRING,
    salary DOUBLE
);

In [0]:
%sql
INSERT INTO dev.demo.demo7 VALUES
(1, 'Rahul', 50000),
(2, 'Amit', 60000);

In [0]:
%sql
SELECT * FROM dev.demo.demo7;

In [0]:
%sql
UPDATE dev.demo.demo7
SET salary = 55000
WHERE id = 1;

In [0]:
%sql
INSERT INTO dev.demo.demo7 VALUES
(3, 'Neha', 70000);

### Check History

In [0]:
%sql
DESCRIBE HISTORY dev.demo.demo7;

### COPY INTO incremental ingestion

In [0]:
%sql
CREATE table if not exists dev.demo.sales_bronze;

In [0]:
%sql
create volume if not exists dev.demo.raw

In [0]:
%sql
copy into dev.demo.sales_bronze
from "/Volumes/dev/demo/raw/sales/"
fileformat = csv
format_options(
    "header" = "true"
)
copy_options(
    'mergeSchema' = 'true'
)

In [0]:
%sql
select count(*) from dev.demo.sales_bronze

### Delta Time Travel

In [0]:
%sql
describe history dev.demo.sales_bronze

### VERSION AS OF

In [0]:
%sql
select * from dev.demo.sales_bronze version as of 1;

### TIMESTAMP AS OF

In [0]:
%sql
select * from dev.demo.sales_bronze timestamp as of "2026-08-27T07:03:41.000+00:00";

# **Intermediate Tasks**

## Schema evolution

In [0]:
%sql
create or replace table dev.demo.schema_demo(
    id int,
    name string,
    salary int
);

In [0]:
%sql
insert into dev.demo.schema_demo values
(1, "raj", 1000000),
(2, "ayush", 2000000),
(3, "nilesh", 3000000);

In [0]:
%sql
select * from dev.demo.schema_demo

In [0]:
%sql
describe dev.demo.schema_demo

### Create new data with a new column

In [0]:
new_data = [
    (4, "neha", 150000, "IT"),
    (5, "raju", 120000, "HR")
]

new_df = spark.createDataFrame(new_data, ["id", "name", "salary", "department"])

In [0]:
new_df.display()

## Schema Evolution

### Use mergeSchema

In [0]:
new_df.write.mode("append").option("mergeSchema", "true").saveAsTable("dev.demo.schema_demo")

In [0]:
%sql
select * from dev.demo.schema_demo

## use overwriteSchema

### now, update type of salary from int to double using overwriteschema.

In [0]:
add_data = [
    (1, "raj", 55000.50, "IT"),
    (2, "ayush", 62000.75, "Finance"),
    (3, "nilesh", 700000.25, "doctor"),
    (4, "neha", 81000.90, "hr"),
    (5, "raju", 81000.90, "Finance")
]
add_df = spark.createDataFrame(add_data, ["id", "name", "salary", "department"])

In [0]:
add_df.printSchema()

In [0]:
add_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("dev.demo.schema_demo")

In [0]:
spark.table("dev.demo.schema_demo").printSchema()

## Merge Schema vs Overwrite Schema

| Feature | `mergeSchema` | `overwriteSchema` |
|---|---|---|
| Purpose | Adds new columns or merges compatible schema changes | Replaces the existing table schema |
| Common Write Mode | `append` | `overwrite` |
| Existing Data | Preserved | Existing data is overwritten by the new write |
| Typical Use | Add a new column to an existing Delta table | Change/replace an existing column's data type or schema |
| Example | Add `department` column | Change `salary` from `INT` to `DOUBLE` |
| Risk | Lower, because existing data is preserved | Higher, because data is overwritten |
| Syntax | `.option("mergeSchema", "true")` | `.option("overwriteSchema", "true")` |

In [0]:
input_path = "/Volumes/dev/demo/raw/input/"
checkpoint_path = "/Volumes/dev/demo/raw/checkpoint/"

### Create the target Delta table through Auto Loader

In [0]:
df_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", checkpoint_path + "/schema")
    .option("header", "true")
    .load(input_path)
)

In [0]:
df_stream.writeStream.option("checkpointLocation", checkpoint_path).outputMode("append").trigger(availableNow=True).toTable("dev.demo.sales_stream")

In [0]:
# df_stream.display(checkpointLocation=checkpoint_path + "/display")

## Task 6 — RESTORE a Delta Table

### First check your table

In [0]:
%sql
select * from dev.demo.schema_demo

In [0]:
%sql
DESCRIBE dev.demo.schema_demo;

### Check the current Delta history

In [0]:
%sql
DESCRIBE HISTORY dev.demo.schema_demo;

### intentionally bad change

In [0]:
bad_data = [
    (1, "Rahul", "WRONG", "IT"),
    (2, "Amit", "WRONG", "Finance"),
    (3, "Neha", "WRONG", "IT"),
    (4, "Ravi", "WRONG", "Finance")
]

df_bad = spark.createDataFrame(
    bad_data,
    ["id", "name", "salary", "department"]
)

df_bad.printSchema()

In [0]:
df_bad.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("dev.demo.schema_demo")

In [0]:
%sql
describe dev.demo.schema_demo;

In [0]:
%sql
describe history dev.demo.schema_demo;

In [0]:
%sql
select * from dev.demo.schema_demo version as of 3

### RESTORE the table

In [0]:
%sql
RESTORE TABLE dev.demo.schema_demo
TO VERSION AS OF 3;

In [0]:
%sql
select * from dev.demo.schema_demo

### Task 6 — RESTORE

I intentionally introduced a bad schema change by changing the `salary`
column from `DOUBLE` to `STRING` using `overwriteSchema`.

Using `DESCRIBE HISTORY`, I identified the last known good version of the
Delta table. I verified the data and schema using Time Travel with
`VERSION AS OF`, and then restored the table using `RESTORE TABLE`.

After the restore, the table returned to the schema and data state of the
selected historical version.

The versions created after the restore point are not deleted from the
Delta transaction history. Instead, RESTORE creates a new version that
represents the restore operation.

## **Advanced Tasks** 

## Task 7 — Ingestion Pattern Comparison

| Ingestion Pattern | Cost | Latency | Operational Complexity | Best Use |
|---|---|---|---|---|
| Batch CTAS | Low | High | Low | Scheduled batch processing |
| COPY INTO | Low | Medium | Low | Incremental file loading |
| Auto Loader | Efficient | Low | Medium | Continuously or unpredictably arriving files |
| Lakeflow Declarative Pipelines | Depends on configuration | Low | Lower operational burden | Managed production data pipelines |

### Recommendation

For Cyntexa's use case, I recommend **Auto Loader**.

The source files arrive unpredictably throughout the day, so a continuously running or incrementally triggered file-ingestion pattern is more suitable than a traditional batch approach. Auto Loader is designed for incremental file ingestion and can detect and process new files as they arrive.

Although Auto Loader introduces some additional operational concepts such as checkpoints and schema management, it provides a good balance between low latency, incremental processing, and operational reliability for this use case.

**Conclusion:** Auto Loader is the most appropriate choice for Cyntexa when files arrive unpredictably throughout the day.

### Task 8 — Recovery Runbook

%md
# Task 8 — Recovery Runbook

## Scenario

A bad file has corrupted the Silver Delta table at 2:00 AM.

The objective is to identify the last known good version of the Delta table,
validate it using Delta Time Travel, restore the table, and verify the recovery.

---

## Step 1 — Check the Current Table

First, inspect the current data to confirm that the table contains incorrect
or corrupted data.

```sql
SELECT *
FROM dev.demo.schema_demo

In [0]:
%sql
select * from dev.demo.schema_demo

In [0]:
%sql
describe dev.demo.schema_demo

In [0]:
%sql
describe history dev.demo.schema_demo;

### Verify the last good version

In [0]:
%sql
select * from dev.demo.schema_demo version as of 3;

### Restore good version of table 

In [0]:
%sql
RESTORE TABLE dev.demo.schema_demo
TO VERSION AS OF 3;

### Validate the recovery

In [0]:
%sql
select * from dev.demo.schema_demo;

## Task 9 — Data Freshness Report

### Create the freshness report

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

history_df = spark.sql("""
    DESCRIBE HISTORY dev.demo.schema_demo
""")

display(history_df)

### Calculate time between updates

In [0]:
window_spec = Window.orderBy("timestamp")

freshness_df = (
    history_df
    .select("version", "timestamp", "operation")
    .withColumn(
        "previous_timestamp",
        F.lag("timestamp").over(window_spec)
    )
    .withColumn(
        "minutes_since_previous_update",
        (
            F.col("timestamp").cast("long") -
            F.col("previous_timestamp").cast("long")
        ) / 60
    )
    .orderBy("timestamp")
)

display(freshness_df)

### Calculate the average update frequency

In [0]:
avg_freshness = (
    freshness_df
    .filter(F.col("minutes_since_previous_update").isNotNull())
    .agg(
        F.avg("minutes_since_previous_update")
        .alias("average_minutes_between_updates")
    )
)

display(avg_freshness)

### Find the fastest and slowest updates

In [0]:
freshness_summary = (
    freshness_df
    .filter(F.col("minutes_since_previous_update").isNotNull())
    .agg(
        F.avg("minutes_since_previous_update").alias("average_minutes"),
        F.min("minutes_since_previous_update").alias("minimum_minutes"),
        F.max("minutes_since_previous_update").alias("maximum_minutes")
    )
)

display(freshness_summary)

### Validate an SLA claim

In [0]:
sla_minutes = 30

sla_result = (
    freshness_summary
    .withColumn(
        "sla_minutes",
        F.lit(sla_minutes)
    )
    .withColumn(
        "sla_status",
        F.when(
            F.col("average_minutes") <= F.col("sla_minutes"),
            "SLA Met"
        ).otherwise("SLA Not Met")
    )
)

display(sla_result)

# Task 9 — Data Freshness Report

## Objective

Use Delta Lake `DESCRIBE HISTORY` to analyze how frequently the table is
actually updated and validate the data freshness SLA claimed by the
business stakeholder.

## Approach

1. Retrieve the Delta table history using `DESCRIBE HISTORY`.
2. Extract the version and timestamp of each table operation.
3. Use the previous timestamp to calculate the time between updates.
4. Calculate the average, minimum, and maximum update intervals.
5. Compare the actual update frequency against the business SLA.
6. Report whether the SLA is met.

## Table

The table analyzed for this report is:

`dev.demo.schema_demo`